### This notebook is for the one-time loading of USGS basin polygon locations into the teehr data warehouse
- Use location ID prefix: `usgsbasin-<usgs gage id>`
- NWM 3.0 retrospective RAINRATE (AORC but maybe slightly different?) will be saved as `primary_timeseries`
- No crosswalk data was added at this time

In [ ]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

In [ ]:
BASINS_FILEPATH = Path("/data/common/geometry/usgs_basins/usgsbasin_geometry_04202026.parquet")

In [ ]:
%%time
spark = create_spark_session(
    aws_profile="default",
)
ev = teehr.RemoteReadWriteEvaluation(spark=spark)

#### Locations

In [ ]:
gdf = gpd.read_parquet(BASINS_FILEPATH)  # NOTE: We should include EPSG/CRS in the locations table? (table properties or an extra field?)
gdf.crs

In [ ]:
gdf.head()

In [ ]:
gdf.index.size

In [ ]:
gdf["geometry"] = gdf.geometry.make_valid()

Append to the locations table in the warehouse

In [ ]:
%%time
ev.locations.load_dataframe(
    df=gdf,
    write_mode="append"
)

In [ ]:
gdf["properties"].iloc[0]

### Also need to link the basin location IDs to themselves in the crosswalk table
This supports using forcing assim as primary and forcing forecasts as secondary at the same basin ID

In [ ]:
df = ev.locations.filter([
    {
        "column": "id",
        "operator": "like",
        "value": "usgsbasin-%"
    }
]).to_sdf().select("id").toPandas()

In [ ]:
xwalk_df = pd.DataFrame(
    {
        "primary_location_id": df["id"].tolist(),
        "secondary_location_id": df["id"].tolist()
    }    
)
xwalk_df.head()

In [ ]:
%%time
ev.location_crosswalks.load_dataframe(
    df=xwalk_df,
    write_mode="append"
)